<a href="https://colab.research.google.com/github/mugalan/introduction-to-statistical-learning/blob/main/linear_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Overview

Gaussian Process Regression (GPR) is a Bayesian approach to regression where we treat the underlying function as a random variable. Unlike traditional regression, which searches for a single "best" set of parameters (like weights in a neural network), GPR maintains a distribution over all possible functions that could fit the data.

The problem can be broken down into three conceptual layers: **The Latent Process**, **The Noisy Observation**, and **The Optimal Inference**.

---

**The Latent Process (The "Signal"):**

We assume there exists an underlying, "clean" process $X_g$ (where $g$ could be time, space, or any input dimension). We define this process as a **Gaussian Process**:

* For any set of points, the values of $X_g$ follow a Multivariate Gaussian distribution.
* This process is entirely defined by its **Mean Function** $\mu_g$ (the average behavior) and its **Kernel** $\kappa(g, g')$ (which defines the "shape" or smoothness of the functions).

---

**The Noisy Observation (The "Measurement"):**

In the real world, we rarely see the clean signal $X_g$ directly. Instead, we observe $Y_g$, which is the signal corrupted by random noise $\nu_g$.

* **The Problem:** We have a collection of $n$ noisy data points $\mathscr{Y}_n = [y_{g_1}, y_{g_2}, \dots, y_{g_n}]$.
* **The Objective:** Given these noisy points, how do we reconstruct the most likely version of the clean signal $X_g$ and, crucially, how much should we trust that reconstruction?

---

**Summary:**

The goal of Gaussian Process Regression is to take **noisy, discrete observations** and produce a **continuous, probabilistic function**.

It doesn't just provide a line; it provides a "corridor of probability" that tells us not only **what** we think the value is, but **how sure we are** based on the proximity of our observations and our prior assumptions about the signal's smoothness.

> **In a sentence:** GPR is the process of using Bayesian inference to "squash" an infinite space of possible functions into a narrow band of likely functions that honor our noisy data.

# 2. Gaussian Process Regression

## 2.1. The 1-D Formulation

Let ${G}$ be some set. Consider a $1$-dimensional Gaussian process $X_g\sim \mathscr{N}(\mu_g,\kappa(g,g))$ where $g\in G$ and $\kappa:G\times G \mapsto \mathbb{R}$ is the covariance (also known as the kernel)
$$\mathbb{E}\left((X_g-\mu_g)(X_{g'}-\mu_{g'})\right)=\kappa(g,g').$$

Let $Y_g$ be another stochastic process that satisfies
\begin{align}
Y_g&=X_g+\nu_g,
\end{align}
where $\nu_g\sim \mathscr{N}(\mu,\sigma_m^2)$.
Assume that $\nu_g\perp X_{g'}$ for all $g,g'\in G$ and that
$$
\mathrm{Cov}(\nu_g,\nu_{g'})= \sigma_m^2 \delta_{gg'}
$$

This represents a noisy sampling of $X_g$. If one samples the stochastic process $Y_g$ at different $g$ and have the set of observations ${y}_{n}\triangleq [y_{g_1},y_{g_2},\cdots,y_{g_n}]^T$, the Gaussian Process (GP) regression is the problem of estimating $X_g$ given the observations $y_n$.


Consider the multidimensional random variable
$\mathscr{Y}_n\triangleq[Y_{g_1},Y_{g_2},\cdots,Y_{g_n}]^T$ and $\mathscr{X}_n\triangleq[X_{g_1},X_{g_2},\cdots,X_{g_n}]^T$. Then
$$\mathscr{Y}_n\sim  \mathscr{N}(\mu_{\mathscr{Y}_n},K_n+\sigma_m^2I)$$
where, $\mu_{\mathscr{Y}_n}\triangleq[\mu_{g_1}+\mu,\mu_{g_2}+\mu,\cdots,\mu_{g_n}+\mu]^T$ and $K_n$ is the $n\times n$ covariance matrix
\begin{align*}
K_n&\triangleq \mathbb{E}\left((\mathscr{X}_n-\mu_{\mathscr{X}_n})(\mathscr{X}_n-\mu_{\mathscr{X}_n})^T\right)=
\begin{bmatrix}
\kappa(g_1,g_1) & \kappa(g_1,g_2) & \cdots & \kappa(g_1,g_n)\\
\vdots & \vdots & \ddots &\vdots\\
\kappa(g_n,g_1) & \kappa(g_n,g_2) & \cdots & \kappa(g_n,g_n)
\end{bmatrix},
\end{align*}
where
$\mu_{\mathscr{X}_n}\triangleq[\mu_{g_1},\mu_{g_2},\cdots,\mu_{g_n}]^T.$
Let
$$
k\triangleq
\begin{bmatrix}\kappa(g_1,g)&\kappa(g_2,g) & \cdots &\kappa(g_n,g)
\end{bmatrix}^T,
$$
then we have that
$$
\mathrm{Cov}(\mathscr{Y}_n,X_g)=k
$$
and hence that
\begin{align*}
\left(\begin{array}{c} X_{g}\\ \mathscr{Y}_n
\end{array}\right)\sim\mathscr{N}\left(\begin{bmatrix}  \mu_{g} \\ \mu_{\mathscr{Y}_n}
\end{bmatrix},\begin{bmatrix} \kappa(g,g) & k^T \\ k & K_n+\sigma^2_m
I \end{bmatrix}\right).
\end{align*}

Thus the Gaussian Process (GP) regression problem boils down to finding $\mathbb{E}[X_{g}|\mathscr{Y}_n=y_n]$.


From the results shown in this note on [Multi-Variate-Gaussians](https://github.com/mugalan/introduction-to-statistical-learning/blob/main/Multivariate_Gaussian_Distributions.ipynb) we see that the conditional random variable satisfies

$$(X_g \mid \mathscr{Y}_n = y_n) \sim \mathscr{N}\Big(\mu_g + k^T(K_n+\sigma_m^2I)^{-1}(y_n - \mu_n), \quad \kappa(g,g) - k^T(K_n+\sigma_m^2I)^{-1}k\Big).$$


Thus the optimal estimate of $X_g$ given the observations $\mathscr{Y}_n=y_n$ is given by:
$$\mathbb{E}[X_g \mid \mathscr{Y}_n=y_n] = \mu_g + k^T(K_n+\sigma_m^2I)^{-1}(y_n - \mu_{\mu_{\mathscr{Y}_n}}),$$
and the uncertainty is given by
$$\mathrm{Var}(X_g \mid \mathscr{Y}_n=y_n)=\kappa(g,g) - k^T(K_n+\sigma_m^2I)^{-1}k.$$

Thus the Gaussian Process Regression problem **boils down to the estimation of an appropriate Kernal (Covariance) $\kappa(g,g')$.**

> While the conditioning identities above provide the mechanism for prediction, the **Kernel** encodes the fundamental physics/logic of the underlying process. Selecting $\kappa(g,g')$ is equivalent to defining the hypothesis space of the regression.

## 2.2. The General multivariate formulation

Let $G$ be some index set. Consider a $q$-dimensional Gaussian process

$$
X_g\in\mathbb R^q,\qquad g\in G,
$$

with mean function

$$
\mu_g \triangleq \mathbb E[X_g]\in\mathbb R^q
$$

and matrix-valued covariance kernel

$$
\kappa:G\times G\to \mathbb R^{q\times q},
$$

defined by

$$
\kappa(g,g')
\triangleq
\mathbb E\left[
(X_g-\mu_g)(X_{g'}-\mu_{g'})^T
\right].
$$

Thus, for every $g,g'\in G$,

$$
\kappa(g,g')\in\mathbb R^{q\times q}.
$$

The diagonal block $\kappa(g,g)$
is the covariance matrix of the random vector $X_g$.


Now let $Y_g\in\mathbb R^q$ be a noisy observation of (X_g), satisfying

$$
Y_g=X_g+\nu_g,
$$

where

$$
\nu_g\sim \mathscr N(\mu_\nu,\Sigma_\nu),
\qquad
\mu_\nu\in\mathbb R^q,\quad
\Sigma_\nu\in\mathbb R^{q\times q}.
$$

Assume that the noise variables are independent across sample locations and independent of the latent process (X_g). That is,

$$
\operatorname{Cov}(\nu_g,\nu_{g'})=\Sigma_\nu \delta_{g,g'}.
$$

Here $\delta_{g,g'}$ is the Kronecker delta.


Suppose we sample the noisy process at points

$$
g_1,\dots,g_n\in G.
$$

Define the stacked random vectors

$$
\mathscr X_n
\triangleq
\begin{bmatrix}
X_{g_1}\\
X_{g_2}\\
\vdots\\
X_{g_n}
\end{bmatrix}
\in\mathbb R^{nq},
\qquad
\mathscr Y_n
\triangleq
\begin{bmatrix}
Y_{g_1}\\
Y_{g_2}\\
\vdots\\
Y_{g_n}
\end{bmatrix}
\in\mathbb R^{nq}.
$$

Also define the stacked mean vectors

$$
\mu_{X,n}
\triangleq
\begin{bmatrix}
\mu_{g_1}\\
\mu_{g_2}\\
\vdots\\
\mu_{g_n}
\end{bmatrix}
\in\mathbb R^{nq},
$$

and

$$
\mu_{Y,n}
\triangleq
\begin{bmatrix}
\mu_{g_1}+\mu_\nu\\
\mu_{g_2}+\mu_\nu\\
\vdots\\
\mu_{g_n}+\mu_\nu
\end{bmatrix}
=
\mu_{X,n}+\mathbf 1_n\otimes \mu_\nu.
$$

The latent covariance matrix is now a block matrix

$$
K_n
\triangleq
\operatorname{Cov}(\mathscr X_n,\mathscr X_n)
=
\begin{bmatrix}
\kappa(g_1,g_1) & \kappa(g_1,g_2) & \cdots & \kappa(g_1,g_n)\\
\kappa(g_2,g_1) & \kappa(g_2,g_2) & \cdots & \kappa(g_2,g_n)\\
\vdots & \vdots & \ddots & \vdots\\
\kappa(g_n,g_1) & \kappa(g_n,g_2) & \cdots & \kappa(g_n,g_n)
\end{bmatrix}
\in\mathbb R^{nq\times nq}.
$$

Since the observation noise is independent across samples, the noisy observation vector satisfies

$$
\mathscr Y_n
\sim
\mathscr N
\left(
\mu_{Y,n},
K_n + I_n\otimes \Sigma_\nu
\right).
$$

This is the multivariate-output analogue of the scalar GP observation model.


Now consider estimating the latent vector $X_g\in\mathbb R^q$ at a new location $g$. The joint distribution of $X_g$ and the observed vector $\mathscr Y_n$ is

$$
\begin{bmatrix}
X_g\\
\mathscr Y_n
\end{bmatrix}
\sim
\mathscr N
\left(
\begin{bmatrix}
\mu_g\\
\mu_{Y,n}
\end{bmatrix},
\begin{bmatrix}
\kappa(g,g) & K_{g,n}\\
K_{n,g} & K_n+I_n\otimes \Sigma_\nu
\end{bmatrix}
\right),
$$

where

$$
K_{g,n}
\triangleq
\begin{bmatrix}
\kappa(g,g_1) & \kappa(g,g_2) & \cdots & \kappa(g,g_n)
\end{bmatrix}
\in\mathbb R^{q\times nq},
$$

and

$$
K_{n,g}
=K_{g,n}^T
\begin{bmatrix}
\kappa(g_1,g)\\
\kappa(g_2,g)\\
\vdots\\
\kappa(g_n,g)
\end{bmatrix}
\in\mathbb R^{nq\times q}.
$$

Therefore, the multivariate GP regression problem is to compute

$$
\mathbb E[X_g\mid \mathscr Y_n=y_n].
$$

Using the conditional Gaussian formula,

$$
\mathbb E[X_g\mid \mathscr Y_n=y_n]=
\mu_g
+
K_{g,n}
\left(
K_n+I_n\otimes\Sigma_\nu
\right)^{-1}
\left(
y_n-\mu_{Y,n}
\right)
$$
and the posterior covariance is

$$
\operatorname{Cov}(X_g\mid \mathscr Y_n=y_n)
=\kappa(g,g)-
K_{g,n}
\left(
K_n+I_n\otimes\Sigma_\nu
\right)^{-1}
K_{n,g}.
$$

This is the direct multivariate-output version of standard GP regression.

# 3. The Determination of a Suitable Kernel

1. The Kernel: Defining the Function Space
The kernel $\kappa(g, g')$ encodes our prior assumptions about the data. It is a similarity measure; if $\kappa(g, g')$ is large, the model assumes $X_g$ and $X_{g'}$ are highly correlated.
**Common Kernel Choices:**

* **Radial Basis Function (RBF) / Squared Exponential:**
The "gold standard" for smooth processes. It assumes the function is infinitely differentiable.

$$\kappa(g, g') = \sigma_f^2 \exp\left( -\frac{\|g - g'\|^2}{2\ell^2} \right)$$


* **Matérn Kernel:**
More realistic for physical or financial data where "jaggedness" exists. It has a parameter $\nu$ that controls smoothness (e.g., $\nu=1.5$ or $2.5$).
* **Periodic Kernel:**
Used for data with repeating cycles (e.g., daily temperature or seasonal sales).

$$\kappa(g, g') = \sigma_f^2 \exp\left( -\frac{2\sin^2(\pi|g-g'|/p)}{\ell^2} \right)$$



---

2. The Hyperparameter Optimization Problem
A kernel is rarely "fixed." It usually depends on a vector of hyperparameters $\theta$, such as the **length-scale** ($\ell$), **signal variance** ($\sigma_f^2$), and **noise variance** ($\sigma_m^2$).
\
\
To find the "best" $\theta$, we don't look at the prediction error. Instead, we maximize the **Log-Marginal Likelihood (LML)**. This is a Bayesian way of asking: *"How likely is it that our observed data $\mathbf{y}_n$ was generated by a GP with these specific settings?"*
\
\
**The Objective Function**
The marginal distribution is given by:
$$\mathscr{Y}_n \sim \mathscr{N}(\mu_{\mathscr{Y}_n}, K_n + \sigma_m^2 I)$$
\
\
The **Marginal Likelihood** is simply the Probability Density Function (PDF) of this multivariate normal distribution evaluated at your actual observed data points $y_n$.
\
$$p(y_n \mid g, \theta) = \frac{1}{\sqrt{(2\pi)^n |K_n + \sigma_m^2 I|}} \exp\left( -\frac{1}{2}(y_n - \mu_{\mathscr{Y}_n})^T (K_n + \sigma_m^2 I)^{-1} (y_n - \mu_{\mathscr{Y}_n}) \right)$$
Taking the **natural log** ($\ln$) of this density gives you the
**Log-Marginal Likelihood:**
$$\log p(y_n \mid g, \theta) = \underbrace{-\frac{1}{2}(y_n - \mu_{\mathscr{Y}_n})^T (K_n + \sigma_m^2 I)^{-1} (y_n - \mu_{\mathscr{Y}_n})}_{\text{Data Fit}} - \underbrace{\frac{1}{2} \log |K_n + \sigma_m^2 I|}_{\text{Complexity Penalty}} - \underbrace{\frac{n}{2} \log 2\pi}_{\text{Normalization}}$$
\
**The "Ockham's Razor" Balance**
* **Data Fit:** This term gets better (larger) as the model fits the data points more closely. It encourages "wigglier" kernels that go exactly through the points.
* **Complexity Penalty:** This term gets worse (smaller) as the kernel becomes more complex or the correlation matrix becomes more diverse. It encourages "simpler," smoother kernels.
\
**The Optimization Task:**
We solve for $\theta^*$ using gradient-based optimizers (like L-BFGS-B):
$$\theta^* = \arg \max_{\theta} \log p(y_n \mid g, \theta)$$

---

**Summary of the Workflow**

1. **Select Kernel Structure:** (e.g., RBF + White Noise).
2. **Initialize $\theta$:** Guess starting values for $\ell$, $\sigma_f^2$, and $\sigma_m^2$.
3. **Optimize:** Maximize the LML to find $\theta^*$. This "tunes" the model to the specific "wiggliness" and "noise" of your dataset.
4. **Predict:** Plug $\theta^*$ into your conditional mean and variance formulas to perform regression.

# 4. Example

The following Python code provides a practical implementation of the **Latent Signal Extraction** problem. Specifically, it demonstrates how GPR can reconstruct a hidden, continuous truth from a small set of "corrupted" data points.

Here is a breakdown of the specific problems the code addresses:

#### 1. The Denoising Problem (Filtering)

In the code, the `y_train` values are generated using a sine wave plus random Gaussian noise.

* **The Code's Solution:** By setting the noise parameter `alpha=0.1`, the model recognizes that the black dots are not the "absolute truth."
* **The Result:** The predictive mean (the green line) provides a smooth, filtered version of the data, effectively separating the **signal** (the underlying physics) from the **noise** (random fluctuations).

#### 2. The Interpolation Problem (Gap Filling)

The training data has significant gaps (e.g., no data exists between $t=3$ and $t=5$).

* **The Code's Solution:** The GPR uses the **Kernel** to calculate the correlation between the known points and the empty spaces.
* **The Result:** It "fills in the blanks" with the most statistically likely path. Notice how the green line naturally follows a curve even where no data points exist.

#### 3. The Uncertainty Quantification Problem

Traditional regression (like a polynomial fit) would give you a line but would be "overconfident"—it wouldn't tell you where it might be wrong.

* **The Code's Solution:** The code calculates the standard deviation $\sigma_g$ for every point along the x-axis.
* **The Result:** The **95% Confidence Interval** (the shaded cloud). You will notice that in the "gaps" between data points, the cloud gets wider. This visually represents the model saying: *"I am making a guess here, but because I haven't seen data in this region, my confidence is lower."*

#### 4. The Hyperparameter "Learning" Problem

The code does not manually set how "wiggly" the curve should be.

* **The Code's Solution:** It uses the **Log-Marginal Likelihood (LML)** optimization we discussed.
* **The Result:** Before fitting, the kernel started with a generic `length_scale=1.0`. After `gp.fit()`, the optimizer looked at the spacing of your data and updated this to an **optimal length-scale**. This ensures the curve is neither too "jittery" nor too "stiff" for the specific data provided.

---

#### Summary for your Documentation

> "The provided Python implementation solves the problem of **Probabilistic Non-Parametric Regression**. It transforms seven noisy, discrete observations into a continuous predictive distribution. The code demonstrates that the GPR is not merely a curve-fitter, but a **risk-aware estimator** that balances empirical data fit with theoretical smoothness assumptions."

In [ ]:
import numpy as np
import plotly.graph_objects as go
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

# 1. Generate Synthetic "Financial" Data
# Imagine this is a normalized stock price over 10 days
np.random.seed(42)
X_train = np.array([1, 3, 5, 6, 7, 8, 9.5]).reshape(-1, 1)
y_train = np.sin(X_train).ravel() + np.random.normal(0, 0.15, X_train.shape[0])

# 2. Define the Kernel (Structure)
# We use an RBF kernel (smoothness) with a constant scaling factor
# Bounds allow the optimizer to "search" for the best hyperparameters
kernel = C(1.0, (1e-3, 1e3)) * RBF(1.0, (1e-2, 1e2))

# 3. GP Regression & Hyperparameter Optimization
# n_restarts_optimizer handles the non-convex LML surface
gp = GaussianProcessRegressor(kernel=kernel, alpha=0.1, n_restarts_optimizer=10)
gp.fit(X_train, y_train)

# 4. Predict over a dense range for plotting
X_plot = np.linspace(0, 11, 200).reshape(-1, 1)
y_mean, y_std = gp.predict(X_plot, return_std=True)

# 5. Visualize with Plotly
fig = go.Figure()

# Add 95% Confidence Interval (The Uncertainty Cloud)
fig.add_trace(go.Scatter(
    x=np.concatenate([X_plot.ravel(), X_plot.ravel()[::-1]]),
    y=np.concatenate([y_mean - 1.96 * y_std, (y_mean + 1.96 * y_std)[::-1]]),
    fill='toself',
    fillcolor='rgba(0,100,80,0.2)',
    line_color='rgba(255,255,255,0)',
    name='95% Confidence Interval',
    showlegend=True
))

# Add the Predictive Mean (The Estimate)
fig.add_trace(go.Scatter(
    x=X_plot.ravel(), y=y_mean,
    line=dict(color='rgb(0,100,80)'),
    name='Predictive Mean (X_g)'
))

# Add the actual observations (The Noisy Data)
fig.add_trace(go.Scatter(
    x=X_train.ravel(), y=y_train,
    mode='markers',
    marker=dict(color='black', size=10),
    name='Observations (Y_n)'
))

# Formatting
fig.update_layout(
    title=f"GPR Result (Optimized Length-scale: {gp.kernel_.get_params()['k2__length_scale']:.2f})",
    xaxis_title="Time (g)",
    yaxis_title="Price / Value (X_g)",
    template="plotly_white",
    hovermode="x"
)

fig.show()

# 5. Linear Regression (The Linear Kernel)

Below we show that **Linear Regression** is a specific case of Gaussian Process Regression, we simply need to choose a specific **Kernel** $\kappa(g, g')$ that represents a linear relationship.

In linear regression, we assume the underlying function $X_g$ takes the form:
$$X_g = g^T w$$
where $w \in \mathbb{R}^d$ is a random weight 1D-matrix following a generalized Gaussian prior distribution:


$$w \sim \mathscr{N}(\mu_w, \Sigma_w).$$

To accommodate a standard linear model with an explicit intercept, $y = mx + c$, we use a feature augmentation trick. We append a constant $1$ to our input vector and separate the slope and intercept inside the weight vector:$$g \triangleq \begin{bmatrix} \widetilde{g} \\ 1 \end{bmatrix}, \quad w \triangleq \begin{bmatrix} m \\ c \end{bmatrix}$$Substituting these into $X_g = g^T w$ yields $m\widetilde{g} + c$, proving that the intercept is natively handled by the current framework.

Since a linear combination of Gaussian random variables is itself Gaussian, $X_g$ forms a Gaussian Process $X_g \sim \mathscr{N}(\mu_g, \kappa(g,g'))$ where:

* **The Mean Function:**
$$\mu_g \triangleq \mathbb{E}[X_g] = \mathbb{E}[g^T w] = g^T \mu_w$$


* **The Covariance Kernel:**
$$\kappa(g,g') \triangleq \mathbb{E}\left((X_g-\mu_g)(X_{g'}-\mu_{g'})\right) = g^T \Sigma_w g'$$



The observed data point $Y_g$ is the latent linear process corrupted by a biased, independent Gaussian noise variable:

$$Y_g = g^T w + \nu_g$$

where the noise process satisfies:


$$\nu_g \sim \mathscr{N}(\mu, \sigma_m^2), \quad \text{with } \mathbb{E}[\nu_g \nu_{g'}] = \sigma_m^2 \delta_{g,g'}$$


When we collect a dataset of $n$ observations at the input locations $G \triangleq [g_1, g_2, \dots, g_n]^T$, then the $n$-observations are given by
$$\mathscr{Y}_n = G w + \boldsymbol{\nu}_n$$

Where $\boldsymbol{\nu}_n \sim \mathscr{N}(\mu\mathbf{1}, \sigma_m^2 I)$, and $\mathbf{1}$ is an $n \times 1$ vector of ones.

This maps identically back to the final joint distribution, substituting:

* $\mu_g = g^T \mu_w$
* $\mu_{\mathscr{Y}_n} = G\mu_w + \mu\mathbf{1}$
* $K_n = G \Sigma_w G^T$
* $k^T = g^T \Sigma_w G^T$

This formulation completely bridges parametric Bayesian linear regression and the non-parametric GP framework. Substituting the above expressions in the Gaussian Process general equations we have:


**The Optimal Estimate (Predictive Mean):**

$$\mathbb{E}[X_g \mid \mathscr{Y}_n=y_n] = g^T \left( \frac{1}{\sigma_m^2} G^T G + \boldsymbol{\Sigma}_w^{-1} \right)^{-1} \left( \frac{1}{\sigma_m^2} G^T (y_n - \mu \mathbf{1}) + \boldsymbol{\Sigma}_w^{-1} \mu_w \right)$$

**The Uncertainty (Predictive Variance):**

$$\mathrm{Var}(X_g \mid \mathscr{Y}_n=y_n) = g^T \left( \boldsymbol{\Sigma}_w^{-1} + \frac{1}{\sigma_m^2} G^T G \right)^{-1} g$$

## 5.1 Explicit Derivation of the the LR Equations

We start with the derived general Gaussian Process equations for the optimal estimate and uncertainty:


$$\mathbb{E}[X_g \mid \mathscr{Y}_n=y_n] = \mu_g + k^T(K_n+\sigma_m^2I)^{-1}(y_n - \mu_n)$$

$$\mathrm{Var}(X_g \mid \mathscr{Y}_n=y_n)=\kappa(g,g) - k^T(K_n+\sigma_m^2I)^{-1}k$$

Substituting the following expressions

* $\mu_g = g^T \mu_w$
* $\mu_n = G\mu_w + \mu\mathbf{1}$
* $K_n = G \Sigma_w G^T$
* $k^T = g^T \Sigma_w G^T$

we have:

$$\mathbb{E}[X_g \mid \mathscr{Y}_n=y_n] = g^T \mu_w + \underbrace{g^T \Sigma_w G^T (G \Sigma_w G^T + \sigma_m^2 I)^{-1}}_{k^T(K_n + \sigma_m^2 I)^{-1}} (y_n - G \mu_w - \mu \mathbf{1})$$

and

$$\mathrm{Var}(X_g \mid \mathscr{Y}_n=y_n) = g^T \Sigma_w g - \underbrace{g^T \Sigma_w G^T}_{k^T} \underbrace{(G \Sigma_w G^T + \sigma_m^2 I)^{-1}}_{(K_n+\sigma_m^2 I)^{-1}} \underbrace{G \Sigma_w g}_{k}$$

---


To transform these non-parametric GP equations (which invert an $n \times n$ matrix) into parametric linear regression equations (which invert a $d \times d$ feature matrix), we rely on the **Woodbury Matrix Identity**:


$$(A + UCV)^{-1} = A^{-1} - A^{-1}U(C^{-1} + VA^{-1}U)^{-1}VA^{-1}$$

Let's adapt a variant of this identity specifically for the term $(G \Sigma_w G^T + \sigma_m^2 I)^{-1}$. Pulling the matrices through yields two incredibly useful identities:


$$\text{Identity A: } \Sigma_w G^T (G \Sigma_w G^T + \sigma_m^2 I)^{-1} = \left(\Sigma_w^{-1} + \frac{1}{\sigma_m^2} G^T G\right)^{-1} \frac{1}{\sigma_m^2} G^T$$

$$\text{Identity B: } (G \Sigma_w G^T + \sigma_m^2 I)^{-1} = \frac{1}{\sigma_m^2}I - \frac{1}{\sigma_m^2}G\left(\Sigma_w^{-1} + \frac{1}{\sigma_m^2}G^TG\right)^{-1}\frac{1}{\sigma_m^2}G^T$$

---


1. **Substitute the linear components into the GP Mean equation:**

$$\mathbb{E}[X_g \mid \mathscr{Y}_n=y_n] = g^T \mu_w + \underbrace{g^T \Sigma_w G^T (G \Sigma_w G^T + \sigma_m^2 I)^{-1}}_{k^T(K_n + \sigma_m^2 I)^{-1}} (y_n - G \mu_w - \mu \mathbf{1})$$


2. **Apply Identity A to the underlined term:**

$$\mathbb{E}[X_g \mid \mathscr{Y}_n=y_n] = g^T \mu_w + g^T \left(\Sigma_w^{-1} + \frac{1}{\sigma_m^2} G^T G\right)^{-1} \frac{1}{\sigma_m^2} G^T (y_n - G \mu_w - \mu \mathbf{1})$$


3. **Factor out $g^T$ to the far left:**

$$\mathbb{E}[X_g \mid \mathscr{Y}_n=y_n] = g^T \left[ \mu_w + \left(\Sigma_w^{-1} + \frac{1}{\sigma_m^2} G^T G\right)^{-1} \frac{1}{\sigma_m^2} G^T (y_n - \mu \mathbf{1}) - \left(\Sigma_w^{-1} + \frac{1}{\sigma_m^2} G^T G\right)^{-1} \frac{1}{\sigma_m^2} G^T G \mu_w \right]$$


4. **Group the terms containing $\mu_w$ together inside the brackets:**

$$\mathbb{E}[X_g \mid \mathscr{Y}_n=y_n] = g^T \left[ \left( I - \left(\Sigma_w^{-1} + \frac{1}{\sigma_m^2} G^T G\right)^{-1} \frac{1}{\sigma_m^2} G^T G \right) \mu_w + \left(\Sigma_w^{-1} + \frac{1}{\sigma_m^2} G^T G\right)^{-1} \frac{1}{\sigma_m^2} G^T (y_n - \mu \mathbf{1}) \right]$$


5. **Simplify the leading operator on $\mu_w$ by finding a common denominator:**

$$I - \left(\Sigma_w^{-1} + \frac{1}{\sigma_m^2} G^T G\right)^{-1} \frac{1}{\sigma_m^2} G^T G = \left(\Sigma_w^{-1} + \frac{1}{\sigma_m^2} G^T G\right)^{-1} \left[ \left(\Sigma_w^{-1} + \frac{1}{\sigma_m^2} G^T G\right) - \frac{1}{\sigma_m^2} G^T G \right] = \left(\Sigma_w^{-1} + \frac{1}{\sigma_m^2} G^T G\right)^{-1} \Sigma_w^{-1}$$


6. **Substitute this simplified operator back into our expression:**

$$\mathbb{E}[X_g \mid \mathscr{Y}_n=y_n] = g^T \left[ \left(\Sigma_w^{-1} + \frac{1}{\sigma_m^2} G^T G\right)^{-1} \Sigma_w^{-1} \mu_w + \left(\Sigma_w^{-1} + \frac{1}{\sigma_m^2} G^T G\right)^{-1} \frac{1}{\sigma_m^2} G^T (y_n - \mu \mathbf{1}) \right]$$


7. **Factor out the shared matrix inverse $\left(\Sigma_w^{-1} + \frac{1}{\sigma_m^2} G^T G\right)^{-1}$ and multiply through by $\sigma_m^2$:**

$$\mathbb{E}[X_g \mid \mathscr{Y}_n=y_n] = g^T \left( G^T G + \sigma_m^2 \Sigma_w^{-1} \right)^{-1} \left( G^T (y_n - \mu \mathbf{1}) + \sigma_m^2 \Sigma_w^{-1} \mu_w \right)$$

---


1. **Substitute the linear components into the GP Variance equation:**

$$\mathrm{Var}(X_g \mid \mathscr{Y}_n=y_n) = g^T \Sigma_w g - \underbrace{g^T \Sigma_w G^T}_{k^T} \underbrace{(G \Sigma_w G^T + \sigma_m^2 I)^{-1}}_{(K_n+\sigma_m^2 I)^{-1}} \underbrace{G \Sigma_w g}_{k}$$


2. **Apply Identity B to expand the middle inverse term matrix:**

$$\mathrm{Var}(X_g \mid \mathscr{Y}_n=y_n) = g^T \Sigma_w g - g^T \Sigma_w G^T \left[ \frac{1}{\sigma_m^2}I - \frac{1}{\sigma_m^2}G\left(\Sigma_w^{-1} + \frac{1}{\sigma_m^2}G^TG\right)^{-1}\frac{1}{\sigma_m^2}G^T \right] G \Sigma_w g$$


3. **Distribute $g^T \Sigma_w G^T$ and $G \Sigma_w g$ into the brackets:**

$$\mathrm{Var}(X_g \mid \mathscr{Y}_n=y_n) = g^T \Sigma_w g - g^T \left[ \frac{1}{\sigma_m^2} \Sigma_w G^T G \Sigma_w \right] g + g^T \left[ \frac{1}{\sigma_m^2} \Sigma_w G^T G \left(\Sigma_w^{-1} + \frac{1}{\sigma_m^2}G^TG\right)^{-1} \frac{1}{\sigma_m^2} G^T G \Sigma_w \right] g$$


4. **Factor out $g^T$ on the left and $g$ on the right:**

$$\mathrm{Var}(X_g \mid \mathscr{Y}_n=y_n) = g^T \left[ \Sigma_w - \frac{1}{\sigma_m^2} \Sigma_w G^T G \Sigma_w + \frac{1}{\sigma_m^2} \Sigma_w G^T G \left(\Sigma_w^{-1} + \frac{1}{\sigma_m^2}G^TG\right)^{-1} \frac{1}{\sigma_m^2} G^T G \Sigma_w \right] g$$


5. **To clear the clutter inside the brackets, factor out $\Sigma_w$ from the left and right edges:**

$$\mathrm{Var}(X_g \mid \mathscr{Y}_n=y_n) = g^T \Sigma_w \left[ \Sigma_w^{-1} - \frac{1}{\sigma_m^2} G^T G + \frac{1}{\sigma_m^2} G^T G \left(\Sigma_w^{-1} + \frac{1}{\sigma_m^2}G^TG\right)^{-1} \frac{1}{\sigma_m^2} G^T G \right] \Sigma_w g$$


6. **Notice that the center element looks like $A - B + B(A+B)^{-1}B$. This simplifies beautifully down to an exact matrix inverse rule:**

$$\left[ \Sigma_w^{-1} - \frac{1}{\sigma_m^2} G^T G + \frac{1}{\sigma_m^2} G^T G \left(\Sigma_w^{-1} + \frac{1}{\sigma_m^2}G^TG\right)^{-1} \frac{1}{\sigma_m^2} G^T G \right] = \left(\Sigma_w + \sigma_m^2 (G^TG)^{-1}\right)^{-1}$$


7. **Collapsing this back down via the final layer of the Woodbury identity yields the elegant parametric form:**

$$\mathrm{Var}(X_g \mid \mathscr{Y}_n=y_n) = g^T \left( \Sigma_w^{-1} + \frac{1}{\sigma_m^2} G^T G \right)^{-1} g$$

## 5.2 Example

In [ ]:
import numpy as np
import plotly.graph_objects as go
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Hyperparameter, Kernel
from sklearn.preprocessing import StandardScaler

class GeneralizedLinearKernel(Kernel):
    """
    A custom implementation of a Linear Kernel that supports a
    generalized weight prior and an implicit intercept bias.

    Formula: kappa(g, g') = sigma_w^2 * (g^T * g') + sigma_b^2
    """
    def __init__(self, sigma_w=1.0, sigma_b=1.0):
        super().__init__()
        self.sigma_w = sigma_w
        self.sigma_b = sigma_b

    def __call__(self, X, Y=None, eval_gradient=False):
        X = np.atleast_2d(X)
        if Y is None:
            Y = X
        else:
            Y = np.atleast_2d(Y)

        dot_product = np.dot(X, Y.T)
        # Compute Kernel Matrix: (sigma_w^2 * linear_core) + sigma_b^2
        K = (self.sigma_w ** 2) * dot_product + (self.sigma_b ** 2)

        if eval_gradient:
            # dK / d(log(sigma_w)) = 2 * sigma_w^2 * dot_product
            grad_sigma_w = 2 * (self.sigma_w ** 2) * dot_product
            # dK / d(log(sigma_b)) = 2 * sigma_b^2
            grad_sigma_b = 2 * (self.sigma_b ** 2) * np.ones_like(dot_product)

            gradient = np.dstack((grad_sigma_w, grad_sigma_b))
            return K, gradient

        return K

    def get_params(self, deep=True):
        return {"sigma_w": self.sigma_w, "sigma_b": self.sigma_b}

    def set_params(self, **params):
        for parameter, value in params.items():
            setattr(self, parameter, value)
        return self

    @property
    def hyperparameters(self):
        # Broad bounds to allow full optimization freedom
        return [
            Hyperparameter("sigma_w", "numeric", (1e-5, 1e5)),
            Hyperparameter("sigma_b", "numeric", (1e-5, 1e5))
        ]

    def is_stationary(self):
        return False

    def diag(self, X):
        X = np.atleast_2d(X)
        return (self.sigma_w ** 2) * np.sum(X ** 2, axis=1) + (self.sigma_b ** 2)

# 1. Generate Synthetic "Noisy Linear" Data
np.random.seed(42)
n_samples = 15
X_train = np.random.uniform(1, 10, n_samples).reshape(-1, 1)

# True system: True w = 1.5, True intercept/noise mean = 2.0, noise std = 1.2
true_w = 1.5
noise_mean = 2.0
noise_std = 1.2
y_train = (X_train * true_w).ravel() + np.random.normal(noise_mean, noise_std, n_samples)

# Scale X data to ensure clean gradient steps
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)

# 2. Instantiate the Linear GP Model
# alpha represents our observation noise variance (sigma_m^2)
sigma_m_sq = noise_std ** 2
linear_kernel = GeneralizedLinearKernel(sigma_w=1.0, sigma_b=1.0)

# CRITICAL FIX: normalize_y=False lets our sigma_b handle the offset natively
gp = GaussianProcessRegressor(kernel=linear_kernel, alpha=sigma_m_sq, n_restarts_optimizer=10, normalize_y=False)
gp.fit(X_train_scaled, y_train)

# 3. Predict over a dense range for plotting
X_plot = np.linspace(0, 12, 200).reshape(-1, 1)
X_plot_scaled = scaler_X.transform(X_plot)
y_mean, y_std = gp.predict(X_plot_scaled, return_std=True)

# 4. Interactive Visualization with Plotly
fig = go.Figure()

# Add 95% Confidence Interval (The Hourglass Envelope)
fig.add_trace(go.Scatter(
    x=np.concatenate([X_plot.ravel(), X_plot.ravel()[::-1]]),
    y=np.concatenate([y_mean - 1.96 * y_std, (y_mean + 1.96 * y_std)[::-1]]),
    fill='toself',
    fillcolor='rgba(31, 119, 180, 0.15)',
    line_color='rgba(255,255,255,0)',
    name='95% Predictive Confidence',
    showlegend=True
))

# Add the GPR Predictive Mean Line
fig.add_trace(go.Scatter(
    x=X_plot.ravel(), y=y_mean,
    line=dict(color='rgb(31, 119, 180)', width=2.5),
    name='GP Predictive Mean (Linear Regression)'
))

# Add the training observations
fig.add_trace(go.Scatter(
    x=X_train.ravel(), y=y_train,
    mode='markers',
    marker=dict(color='rgb(214, 39, 40)', size=10, line=dict(color='black', width=1)),
    name='Noisy Observations (Y_n)'
))

# Format layout
opt_params = gp.kernel_.get_params()
fig.update_layout(
    title=f"Bayesian Linear Regression via GPR Framework<br><sup>Optimized Parameters: σ_w={opt_params['sigma_w']:.3f}, σ_b={opt_params['sigma_b']:.3f}</sup>",
    xaxis_title="Features / Inputs (g)",
    yaxis_title="Target Value (X_g)",
    template="plotly_white",
    hovermode="x",
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()

# 7. Ordinary Least Squares (OLS)

The derived Gaussian Process results reveal that the classical **Ordinary Least Squares (OLS)** method is simply a restrictive, non-probabilistic limit of this general Bayesian framework. If we strip away the prior assumptions by letting the weight confidence become completely uniform ($\Sigma_w^{-1} \to 0$, meaning an infinite prior variance) and assume that the measurement noise has zero mean ($\mu = 0$), the optimal estimate equation collapses cleanly to:


$$\mathbb{E}[X_g \mid \mathscr{Y}_n=y_n] = g^T (G^T G)^{-1} G^T y_n = g^T \hat{w}_{\text{OLS}}$$


This demonstrates that OLS regression is exactly equivalent to Gaussian Process Regression under a flat prior and a linear kernel. While OLS only provides a single deterministic weight vector $\hat{w}_{\text{OLS}}$ that minimizes empirical squared error, the GPR formulation preserves the full Bayesian landscape. It retains the measurement noise variance $\sigma_m^2$ and the prior structure $\Sigma_w$ as standard regularizers (recovering Ridge Regression/MAP estimation), while simultaneously providing the predictive variance $\mathrm{Var}(X_g \mid \mathscr{Y}_n=y_n)$—a measure of structural uncertainty that classical least squares completely ignores.